# Sigmoid and Mean Squared Error in AI/ML

This notebook introduces two common ideas used when building machine-learning models:

- **Sigmoid** converts any real number into a value between 0 and 1. It is often used to turn a model's score into a probability-like output for binary classification.
- **Mean squared error (MSE)** measures how far a model's predictions are from the expected answers. It is commonly used as a loss function for regression.

We will calculate both with NumPy, then combine them in a tiny binary-classification example. The examples use NumPy only; a training library such as scikit-learn or PyTorch automates much of this calculation in real projects.


In [ ]:
import numpy as np

## 1. The sigmoid function

The sigmoid, also called the logistic function, is

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

`z` is a model score, also called a **logit**. The sigmoid squashes that score into the open interval `(0, 1)`:

- a very negative score gives a value close to `0`;
- a score of `0` gives `0.5`;
- a very positive score gives a value close to `1`.

For binary classification, a model can treat the result as its estimated probability for class `1`. A threshold such as `0.5` can then turn that probability into a class prediction. The threshold is a decision choice and can be changed to match the problem.


In [ ]:
def sigmoid(z):
    """Apply sigmoid element-wise to a number or NumPy array."""
    return 1 / (1 + np.exp(-z))

scores = np.array([-8, -2, 0, 2, 8])
probabilities = sigmoid(scores)

print("scores:       ", scores)
print("sigmoid(scores):", np.round(probabilities, 4))

### Reading sigmoid outputs

A model output of `0.91` means the model assigns a high score to class `1`; `0.08` means it assigns a low score. These values are useful as probabilities only to the extent that the model is well trained and calibrated. Sigmoid alone does not make a model accurate.

In a simple linear model, the score might be `z = w*x + b`, where `w` is a weight and `b` is a bias. With several input features, it is `z = w₁x₁ + w₂x₂ + ... + b`.


In [ ]:
# A simple one-feature model: predict whether a value is above 5.
x = np.array([2.0, 4.0, 6.0, 8.0])
w = 1.0
b = -5.0

logits = w * x + b
predicted_probabilities = sigmoid(logits)
predicted_classes = (predicted_probabilities >= 0.5).astype(int)

print("logits:       ", logits)
print("probabilities:", np.round(predicted_probabilities, 3))
print("classes:      ", predicted_classes)

## 2. The derivative of sigmoid

The derivative is

\[
\sigma'(z) = \sigma(z)(1 - \sigma(z))
\]

It tells us how much the sigmoid output changes when its input changes slightly. This derivative is used by **gradient descent** and **backpropagation** to calculate how model weights should change.

The derivative is largest at `z = 0` and becomes small when the sigmoid is near `0` or `1`. In a deep model, many saturated sigmoid units can make gradients very small, which can slow learning. This is one reason modern deep networks often use other activations in hidden layers, while sigmoid remains common in a binary-classification output layer.


In [ ]:
p = sigmoid(scores)
sigmoid_derivative = p * (1 - p)
print("scores:              ", scores)
print("sigmoid derivative:  ", np.round(sigmoid_derivative, 4))

## 3. Mean squared error (MSE)

For `n` examples, MSE is

\[
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\]

Here `y` contains the actual values and `ŷ` contains the model's predictions. Squaring makes all errors non-negative and gives larger errors more influence. The mean makes the value independent of the number of examples (though its scale still depends on the units of the target).

MSE is a **loss function**: a training algorithm tries to adjust model parameters to make this number smaller. Lower training MSE means predictions are closer to the training targets on average; it does not by itself prove that the model will generalize to new data.


In [ ]:
actual = np.array([3.0, -0.5, 2.0, 7.0])
predicted = np.array([2.5,  0.0, 2.0, 8.0])

squared_errors = (actual - predicted) ** 2
mse = np.mean(squared_errors)

print("actual:        ", actual)
print("predicted:     ", predicted)
print("squared errors:", squared_errors)
print("MSE:           ", mse)

### MSE is especially common for regression

In **regression**, the model predicts a continuous quantity such as price, temperature, or delivery time. MSE is a common loss for fitting these predictions. Its square means a prediction that is 4 units off contributes 16 squared units, while one that is 1 unit off contributes 1.

MSE can be sensitive to outliers because squaring emphasizes large errors. Depending on the task, alternatives such as mean absolute error or Huber loss may be more appropriate.


## 4. Combining sigmoid and MSE for binary classification

For binary labels (`0` or `1`), a model can compute a logit, apply sigmoid to get a value between 0 and 1, and compare that value with the label using MSE:

1. `z = X @ w + b` computes a score for each example.
2. `p = sigmoid(z)` gives a probability-like prediction for class `1`.
3. `MSE = mean((y - p) ** 2)` measures prediction error.

This is useful for understanding how an output activation and a loss fit together. **In most binary-classification training, binary cross-entropy (log loss) is preferred over MSE.** It is designed for probabilistic classification and usually gives more useful gradients when the model is confidently wrong. We use MSE here to connect it to the formula above, not to claim it is the best classification loss for every task.


In [ ]:
# Four one-dimensional examples with binary labels.
X = np.array([[0.0], [1.0], [2.0], [3.0]])
y = np.array([0.0, 0.0, 1.0, 1.0])

w = np.array([0.0])
b = 0.0

logits = X @ w + b
p = sigmoid(logits)
classification_mse = np.mean((y - p) ** 2)
classes = (p >= 0.5).astype(int)

print("probabilities:", np.round(p, 3))
print("predicted classes:", classes)
print("actual labels:    ", y.astype(int))
print("MSE loss:         ", round(classification_mse, 4))

## 5. A tiny gradient-descent example

A model learns by changing its weight and bias to reduce a loss. For sigmoid predictions and MSE, the gradient for each logit uses the chain rule:

\[
\frac{\partial \text{MSE}}{\partial z_i}
= \frac{2}{n}(p_i-y_i) p_i(1-p_i)
\]

Then, for `z = X @ w + b`, the gradients are `X.T @ dL_dz` for the weights and the sum of `dL_dz` for the bias. The code below performs these updates repeatedly. This is a small demonstration of the mechanics; production training uses better initialization, optimizers, validation data, and numerical safeguards.


In [ ]:
X = np.array([[0.0], [1.0], [2.0], [3.0]])
y = np.array([0.0, 0.0, 1.0, 1.0])

w = np.array([0.0])
b = 0.0
learning_rate = 0.5
n = len(y)

for step in range(1000):
    # Forward pass: score, sigmoid prediction, and MSE.
    logits = X @ w + b
    p = sigmoid(logits)
    loss = np.mean((p - y) ** 2)

    # Backward pass: apply the chain rule to get gradients.
    dloss_dlogits = (2 / n) * (p - y) * p * (1 - p)
    grad_w = X.T @ dloss_dlogits
    grad_b = np.sum(dloss_dlogits)

    # Gradient-descent update: move parameters against the gradient.
    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

final_probabilities = sigmoid(X @ w + b)
final_loss = np.mean((final_probabilities - y) ** 2)

print("learned weight:", np.round(w, 3))
print("learned bias:  ", round(b, 3))
print("probabilities: ", np.round(final_probabilities, 3))
print("MSE loss:      ", round(final_loss, 4))

## Summary

- **Sigmoid** turns a score into a number between `0` and `1`; it is often used at the output of a binary classifier.
- **MSE** averages squared differences between actual values and predictions; it is widely used for regression.
- A binary classifier can combine sigmoid with MSE, but binary cross-entropy is generally the more suitable loss for classification.
- During training, gradient descent uses derivatives to adjust weights and biases so the chosen loss decreases.
